# Bonus 02 — Text ReAct Against a Real Model

**Optional | After Lab 1B | Colab CPU | `OPENAI_API_KEY`**

---

Lab 1B Part C built a text-based ReAct loop and broke its parser with five replies we wrote by hand. That showed *how* the regex fails. This notebook shows it failing *on its own*: a real model, three tools, several turns, and a system prompt you can loosen until the model stops following the format.

The text format looks like this, and it is what most agent papers and older frameworks use:

```
Thought: I need memory for a 7B INT4 model.
Action: estimate_memory: 7, int4
PAUSE
Observation: {"gb": 3.5}
Answer: ...
```

Every line of it is a convention the model has to keep. Nothing enforces it. That is the lesson, and the reason production agents use JSON `tool_calls` (Lab 1B Parts A and B).

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} openai python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-4o-mini"

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"Ready — {DEFAULT_MODEL} at {OPENAI_BASE_URL}")

## 1. Three small tools

Ordinary Python functions. Unlike Lab 1B's `calculate`, none of them uses `eval()`: `add_numbers` only adds.


In [ ]:
import json
import re

def estimate_memory(params_b: float, precision: str) -> str:
    table = {"fp32": 4.0, "fp16": 2.0, "int8": 1.0, "int4": 0.5, "nf4": 0.5}
    key = precision.lower()
    if key not in table:
        return json.dumps({"error": f"unsupported precision {precision}"})
    return json.dumps({"params_b": params_b, "precision": key, "gb": round(params_b * table[key], 2)})

def kb_lookup(topic: str) -> str:
    kb = {"qlora": "QLoRA = NF4 quantized base + LoRA adapters. Train ~1% of parameters.",
          "vllm":  "vLLM uses PagedAttention and continuous batching for high throughput serving.",
          "rag":   "RAG retrieves chunks at inference and grounds the prompt. Not a fine-tune."}
    return kb.get(topic.lower().strip(), f"No note stored for {topic!r}. Try qlora, vllm, or rag.")

def add_numbers(a: float, b: float) -> str:
    return json.dumps({"sum": a + b})

print(estimate_memory(7, "int4"))
print(kb_lookup("qlora"))

In text ReAct the model writes arguments as **one string** after the tool name, so we need a parser. `_two` splits on a comma and tries to make numbers. This parser is the fragile part; JSON `tool_calls` (Lab 1B) removed it.

In [ ]:
def _maybe_number(part: str):
    try:
        return float(part)
    except ValueError:
        return part            # precisions like "int4" stay strings

def _two(raw: str):
    parts = [p.strip() for p in raw.split(",")]
    if len(parts) != 2:
        raise ValueError("expected two comma-separated arguments")
    return _maybe_number(parts[0]), _maybe_number(parts[1])

TOOLS = {
    "estimate_memory": lambda raw: estimate_memory(*_two(raw)),
    "kb_lookup":       lambda raw: kb_lookup(raw.strip()),
    "add_numbers":     lambda raw: add_numbers(*_two(raw)),
}

print(TOOLS["estimate_memory"]("7, int4"))
print(TOOLS["add_numbers"]("2, 3.5"))


## 2. The text contract

The system prompt is the entire protocol. If the model adds a period in the wrong place, regex misses the Action. Watch for that.


In [ ]:
SYSTEM = '''You solve questions using this format and nothing else until you can Answer:
Thought: <one sentence>
Action: <tool>: <args>
PAUSE

Tools:
- estimate_memory: <params_b>, <precision>   e.g. estimate_memory: 7, int4
- kb_lookup: <qlora|vllm|rag>                e.g. kb_lookup: qlora
- add_numbers: <a>, <b>                      e.g. add_numbers: 2, 3.5

Precision is one of fp32, fp16, int8, int4, nf4 - never a bare number.

When you have an Observation and can finish:
Thought: <one sentence>
Answer: <final>'''

print("system prompt ready")


## 3. The loop

Max 5 turns. Parse one `Action:` line, run the tool, send `Observation:` back as the next user message.


In [ ]:
ACTION_RE = re.compile(r"^Action:\s*(\w+):\s*(.*)$", re.MULTILINE)

def text_react(question: str, max_turns: int = 5, system: str = SYSTEM) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]
    for turn in range(1, max_turns + 1):
        text = client.chat.completions.create(
            model=DEFAULT_MODEL, temperature=0, messages=messages
        ).choices[0].message.content
        print(f"--- turn {turn} ---")
        print(text)
        messages.append({"role": "assistant", "content": text})
        if re.search(r"^Answer:", text, re.MULTILINE) and not ACTION_RE.search(text):
            return text
        match = ACTION_RE.search(text)
        if not match:
            print("No Action line parsed — this is why JSON tool_calls won.")
            return text
        name, raw = match.group(1), match.group(2)
        fn = TOOLS.get(name)
        obs = fn(raw) if fn else f"Unknown tool {name}"
        print("Observation:", obs)
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "Stopped at max_turns"

print(text_react("How much memory does a 7B INT4 model need, and what is QLoRA in one sentence?"))


**Checkpoint:** you should see `estimate_memory` called with `7, int4`, then an `Answer`. The model may answer the QLoRA half from its own knowledge instead of calling `kb_lookup`. That is a legitimate choice, and watching it decide *not* to use a tool is part of the lesson.

Notice the shape: the loop ran twice, and the second turn only happened because the first one produced an Observation.

## 4. Watch the parser lose

The loop above has a branch you have not seen fire:

```python
if not match:
    print("No Action line parsed — this is why JSON tool_calls won.")
```

The strict `SYSTEM` contract keeps the model in format, so that line never runs. Take the contract away and the loop has nothing to parse. In production that happens the moment a prompt drifts, a model is swapped, or a user's question pulls the model into prose.

In [ ]:
SYSTEM_LOOSE = "You are a helpful assistant. Use tools where they help."

print(text_react("How much memory does a 7B INT4 model need?", system=SYSTEM_LOOSE))

**Checkpoint:** the model answers in plain prose, `ACTION_RE` finds nothing, and the dead branch finally prints. Note what the loop did next: it **returned that text as the answer**. No exception, no warning. The agent quietly stopped being an agent and went back to guessing from memory.

That silence is the argument for structured tools. With JSON `tool_calls` the model either returns a typed call or it does not, and `finish_reason` tells you which. Lab 1B Part C walks through more of these failure shapes, with replies written by hand; this one happened on its own.

**Where this leaves you**

| | This notebook | Lab 1B |
|---|---|---|
| Action | `Action: name: args` | JSON `tool_calls` |
| Parse | Regex | API-typed arguments |
| Failure | Silent: prose returned as the answer | Explicit: `finish_reason`, typed errors |
| Ship it? | No | Yes |


## MCP, in one screen

[Model Context Protocol](https://modelcontextprotocol.io) is a **standard** for exposing tools (and resources) so every agent host does not rewrite `get_flight`. You still write the Python. MCP is the USB port; function calling is the electricity.

You do **not** need to build an MCP server in this course. Know the name for when a vendor says “we speak MCP.”

## Bonus 02 complete

- [ ] The loop called a tool, read the Observation and answered
- [ ] With a loose prompt, the model answered in prose and the loop silently returned it
- [ ] You can say why JSON `tool_calls` fail loudly where text ReAct fails quietly

## Stretch

1. Ask a question that needs two tools, e.g. memory for a 7B model at INT4 *and* at FP16, added together. How many turns does it take? Does it ever skip a tool and do the arithmetic itself?
2. Change `temperature=0` to `0.8` inside `text_react` and run the same question five times. How often does the format break?
3. Rewrite one tool as a JSON tool schema (Lab 1B Part A) and compare the two loops on the same question.

Next: [Bonus 03 — Serve your own model with vLLM](03_vllm_serving.ipynb) (needs a T4), or [Bonus 04 — LiteLLM](04_litellm_gateway.ipynb).